<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/wendland_calibres.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Wendland calibrés sur la géométrie — expérience autonome.
# Nécessite ridge_atom.py dans le même dossier (chargé par exec).
# Régler CFG en bas ; a_mode='impose' valide la chaîne (a=a_true), 'tirage' = vraie tâche.

import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time

# ══════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES
# Cible : deux hyperplans parallèles {P_sep = 0} et {P_sep = ε} (P_sep linéaire),
# label sigmoïde séparant les deux plans. Méthode : experts cylindriques
# (gaussiennes anisotropes) + Wendland anisotropes, UN σ aléatoire par dimension
# et par centre, SANS décroissance géométrique de σ.
# ══════════════════════════════════════════════════════════════════════════════
params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20,
    "seeds": {1: 22476, 2: 12425, 3: 12421, 42: 48124, 43: 24322, 44: 12561, 11: 21365409, 12: 2335},
    "weights":   {0: 0.01, 1: 1, 2: 0.1, 3: 0.0},
    "domain":    (0.0, 50.0),  # domaine [lo, hi]^d où sont tirés les points
    "epsilon":   3,     # écartement des deux plans (valeur de P_sep, pas distance)
    "n_train":   10,
    "n_test":    5000,
    "n_levels":     0,     # nb d'experts cylindriques (gaussiennes anisotropes)
    "n_levels_wnd": 1,     # nb d'experts Wendland anisotropes (support compact)
    "k_loss":    3.0,      # sélection : garder les experts avec loss <= k_loss * meilleure
    "deg_poly_expert": -1, # -1 -> expert polynomial désactivé
    "n_dict":    5000,     # taille du dictionnaire par niveau
    "n_centres": 1000,      # centres retenus par niveau
    "sigma_min": 0.2,      # borne inf des sigma
    "sigma_max": 100,      # borne sup des sigma
    "sigma_mode": "aligned",  # "random" : σ anisotrope tiré au hasard (ancien) |
                              # "aligned": pour chaque centre, on tire N orientations
                              #   σ sur ‖σ‖=1 et on GARDE celle de norme H minimale
                              #   (aligne l'atome sur la variété), puis l'échelle est
                              #   tirée à part -> découvre la direction normale locale.
    "n_orient":  200,      # nb d'orientations testées par centre (mode aligned)
    "candidate_select": "random",  # "random" : sous-échantillon des candidats,
                                   #   indépendant des labels (la pénalité Sobolev
                                   #   trie) | "corr" : ancien top-k par corrélation.
    "lambda_reg": 1e3,    # UNIQUE régularisation Sobolev (Phase 1, sélection, Phase 2)

    "a_label":   2.0,   # amplitude des labels ±a_label (plateaux haut/bas)
    "beta":      1e-3,     # raideur log-cosh ; garder beta*a_label ~ O(1)
    "loss_type": "sigmoid",# "sigmoid" (tanh bornée [0,1], NON convexe) |
                           # "qp" (min ‖u‖_H s.c. marge dure, CONVEXE, stable) |
                           # "hinge2" | "logcosh" | "mse"
    "qp_margin": 1.0,      # marge dure a du QP : y_i u(x_i) >= a
    "beta_sig":  1.0,      # raideur DIRECTE de la sigmoïde : transition sur f ~ ±1/beta.
                           # f est une combinaison de noyaux O(1) -> beta d'ordre 1.
                           # (a_label ne sert plus qu'à hinge2)
    "n_G":       2000,     # points pour la Gram de chaque f_i
    "n_G_H":     5000,     # points pour la Gram finale sur H
}
params["n_unlabeled"]        = 4000 - params["n_train"]
params["train_center_ratio"] = 1.0    # tous les centres tirés côté train+unlabeled












# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d)
                   if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:,j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices)
                   if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:,j]**e
                y += c * term
            return y
        return Pn
    return P

d   = params["n_ambiant"]
eps = params["epsilon"]

# P_sep LINÉAIRE (degré 1) -> les deux surfaces {P_sep=0} et {P_sep=ε} sont de
# vrais hyperplans parallèles, ce qui permet une projection EXACTE (cf. plus bas).
P_sep, idx_sep, coeffs_sep = random_sparse_polynomial(d, 1, 4, seed=99)
P_sep = normalize_polynomial(P_sep, idx_sep, coeffs_sep)

# ══════════════════════════════════════════════════════════════════════════════
# CLOUD sur {Q=0} = {P_sep=0} ∪ {P_sep=ε}   —  PROJECTION EXACTE
#
# BUG CORRIGÉ : l'ancienne version projetait par descente de gradient normalisée
# sur Q = P_sep·(P_sep−ε). Deux défauts :
#   (1) ∇Q = (2·P_sep − ε)·∇P_sep s'annule sur le plan médian {P_sep=ε/2} :
#       les points initialisés près de ce plan restaient PIÉGÉS (gradient nul).
#   (2) Taux de convergence effondré (~0.5 %) -> la boucle de sur-échantillonnage
#       redessinait sans fin et retenait surtout les points du plan le plus
#       facile à atteindre, d'où un déséquilibre ~70/30 entre les deux plans.
#
# Comme P_sep est AFFINE, P_sep(x) = a·x + b, la projection orthogonale d'un
# point x sur l'hyperplan {a·x + b = t} est analytique :
#       x' = x − ((a·x + b − t)/‖a‖²) · a
# On échantillonne dans le cube [lo,hi]^d, on projette exactement, on garde les
# points restés dans le cube. On tire 50/50 entre les deux plans (équilibre exact).
# ══════════════════════════════════════════════════════════════════════════════
DOM_LO, DOM_HI = params["domain"]

_b = P_sep(np.zeros((1, d)))[0]
_a = np.array([P_sep(np.eye(d)[k:k+1])[0] - _b for k in range(d)])
_a2 = float(_a @ _a)
assert _a2 > 0, "P_sep dégénéré (gradient nul)"

def project_to_plane(X, t):
    """Projection orthogonale exacte de X sur l'hyperplan {a·x + b = t}."""
    lam = (X @ _a + _b - t) / _a2
    return X - lam[:, None] * _a

def sample_on_plane(t, n_target, seed, max_tries=1000000):
    """Points uniformes (par rejet) sur {P_sep=t} ∩ [lo,hi]^d, projection exacte."""
    rng = np.random.default_rng(seed); out = []; got = 0; tries = 0
    while got < n_target and tries < max_tries:
        m = max(4 * (n_target - got), 500)
        X  = rng.uniform(DOM_LO, DOM_HI, (m, d))
        Xp = project_to_plane(X, t)
        ok = np.all((Xp >= DOM_LO) & (Xp <= DOM_HI), axis=1)
        g  = Xp[ok]
        if len(g): out.append(g); got += len(g)
        tries += m
    if got < n_target:
        raise RuntimeError(f"plan t={t}: seulement {got}/{n_target} points collectés")
    return np.vstack(out)[:n_target]

def sample_two_planes(n_target, seed):
    """n_target points répartis 50/50 EXACTEMENT sur les deux plans, mélangés."""
    n0 = n_target // 2; n1 = n_target - n0
    X0 = sample_on_plane(0.0, n0, seed)
    X1 = sample_on_plane(eps, n1, seed + 1)
    X  = np.vstack([X0, X1])
    rng = np.random.default_rng(seed + 2)
    perm = rng.permutation(len(X))
    return X[perm]

print("Génération du cloud sur {Q=0} = {P_sep=0} ∪ {P_sep=ε} (projection exacte, 50/50)...")
X_train     = sample_two_planes(params["n_train"],     seed=params["seeds"][42])
X_test      = sample_two_planes(params["n_test"],      seed=params["seeds"][43])
X_unlabeled = sample_two_planes(params["n_unlabeled"], seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_test:{X_test.shape}  X_unlabeled:{X_unlabeled.shape}")

# contrôle d'équilibre + résidu de projection
for nom, Xc in [("train", X_train), ("test", X_test)]:
    p = P_sep(Xc)
    n0 = int((np.abs(p) < eps/2).sum()); n1 = int((np.abs(p-eps) < eps/2).sum())
    resid = np.minimum(np.abs(p), np.abs(p-eps)).max()
    print(f"  {nom:5s} : plan0={n0}  planε={n1}  max|résidu projection|={resid:.2e}")

# ── Cible : label ±a_label selon le plan (plateau haut / plateau bas) ──────────
# CLASSIFICATION : y = +a_label sur {P_sep=ε}, −a_label sur {P_sep=0}. La loss
# log-cosh sature loin de la frontière : seul le côté du seuil (0) compte.
A_LABEL   = params["a_label"]     # marge de hinge2 (échelle de f)
BETA_SIG  = params["beta_sig"]    # raideur directe de la sigmoïde (labels ±1)
def target_label(X):
    return np.where(P_sep(X) > eps/2, A_LABEL, -A_LABEL)

y_train = target_label(X_train); y_test = target_label(X_test)
# Labels ±1 pour les BASELINES (RBF, Ridge poly) : leur échelle naturelle.
# Imposer ±a_label aux baselines biaiserait la comparaison (leur régularisation
# interne est calibrée pour des cibles O(1)). La décision étant sign(f), l'échelle
# du label est un paramètre interne à chaque méthode -> chacun sur la sienne.
y_train_pm1 = np.sign(y_train); y_test_pm1 = np.sign(y_test)

# ── Échelle de travail de la MÉTHODE H ────────────────────────────────────────
# sigmoid : f est ajusté sur des labels ±1 (β règle la marge sur l'échelle de f),
#           donc TOUT le pipeline H (fit, sélection, affichage, MSE) est en ±1.
# hinge2/logcosh/mse : la marge/cible est a_label, pipeline en ±a_label.
if params["loss_type"] in ("sigmoid", "qp"):
    yH_train, yH_test = y_train_pm1, y_test_pm1
else:
    yH_train, yH_test = y_train, y_test
n_pos = int((y_train > 0).sum()); n_neg = int((y_train < 0).sum())
print(f"  y_train: {n_neg} × (−{A_LABEL:g})  |  {n_pos} × (+{A_LABEL:g})   "
      f"(labels ±a_label, décision sign(f))")

s_all_end = None
X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

yH_train = np.sign(target_label(X_train))
yH_test  = np.sign(target_label(X_test))
a_true = _a/np.linalg.norm(_a)
print("cloud prêt. a_true =", a_true.round(3))

# ══════════════════════════════════════════════════════════════════════════════
# WENDLAND CALIBRÉS SUR LA GÉOMÉTRIE  —  expérience autonome
#
# Atome (module ridge_atom) : ellipsoïde à UN axe distingué a (codim 1),
#   φ(x) = ψ(a·(x−c)/σ_a) · W(‖P⊥(x−c)‖/σ_perp),  ψ = plateau, W = Wendland C².
# Pour chaque centre : meilleur a (tirage min norme H) OU a imposé = a_true.
# σ_a = hyperparamètre d'échelle normale (un par expert). Semi-interpolation :
#   min ‖u‖²_H  s.c.  y_i u(x_i) >= marge   (QP convexe), dans une BANDE spectrale.
# Phase 2 : combine les experts (σ_a variés) par le même QP.
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
from scipy.optimize import minimize, LinearConstraint
from google.colab import drive
drive.mount('/content/drive')
import numpy as np

# ══════════════════════════════════════════════════════════════════════════════
# ATOME WENDLAND À PLATEAU ANISOTROPE, distingué par UNE direction a (codim 1)
#
#   φ(x) = ψ( (a·(x−c)) / σ_a )  ·  W( ‖P⊥(x−c)‖ / σ_perp )
#
# - a : direction normale à la variété (unitaire). P⊥ = projection sur a^⊥.
# - ψ : profil 1D À PLATEAU dans la direction a (plat près de 0, puis monte).
#       -> l'atome varie peu le long de a près du centre (tolérance : points un
#          peu à côté du plan) puis décroît. On prend un profil pair, support [-1,1].
# - W : Wendland C² radial dans les 3 directions perpendiculaires (support compact).
# σ_a = |σ| dans la direction a (hyperparamètre d'échelle) ; σ_perp large (les
# directions tangentes : l'atome est étendu le long de la variété).
#
# Dérivées analytiques ordre 1 et 2 (w3=0) pour la norme H de Sobolev.
# ══════════════════════════════════════════════════════════════════════════════

# --- profil 1D à plateau, pair, support [-1,1], C² ---
# base : Wendland 1D (1-|t|)^4 (4|t|+1). Pour un vrai plateau au centre on compose
# avec un "cœur plat" : on décale le début de la décroissance. Simplement : profil
# ρ(t) = 1 sur [-p,p], puis raccord C² vers 0 en |t|=1. On implémente un profil
# polynomial par morceaux ; ici on garde la forme lisse suivante (pair, C²) :
#   ρ(t) = (1-t²)_+ ^3   (bump C², plat au sommet : ρ'(0)=0)  — plateau "doux".
# Pour un plateau plus large, param p contrôle via t -> t/(1-p) recentré.
def _plateau_prof(t, p=0.0):
    """profil pair C², plat au centre (ρ'(0)=0), support |t|<=1. p élargit le plat."""
    if p > 0:
        s = (np.abs(t) - p) / (1.0 - p)
        s = np.clip(s, 0.0, 1.0)
    else:
        s = np.clip(np.abs(t), 0.0, 1.0)
    r = (1.0 - s**2)
    return r**3

def _plateau_prof_d(t, p=0.0):
    """ρ, ρ', ρ'' par rapport à t (chain rule incluse)."""
    at = np.abs(t); sgn = np.sign(t)
    if p > 0:
        s = (at - p) / (1.0 - p)
        inb = (at > p) & (at < 1.0)
        ds_dt = sgn / (1.0 - p)
    else:
        s = at
        inb = at < 1.0
        ds_dt = sgn
    s = np.clip(s, 0.0, 1.0)
    r = (1.0 - s**2)
    rho   = r**3
    drho_ds  = 3*r**2*(-2*s)
    d2rho_ds2 = 6*r*(-2*s)*(-2*s) + 3*r**2*(-2)   # = 24 s² r -6 r²... recompute
    # d/ds [ -6 s (1-s²)² ] = -6(1-s²)² -6s·2(1-s²)(-2s) = -6(1-s²)² +24 s²(1-s²)
    d2rho_ds2 = -6*r**2 + 24*s**2*r
    rho1 = np.where(inb, drho_ds * ds_dt, 0.0)
    rho2 = np.where(inb, d2rho_ds2 * ds_dt**2, 0.0)  # ds_dt const per side -> ok
    return rho, rho1, rho2

# --- Wendland C² radial (perp) : w(r)=(1-r)^4(4r+1), support r<1 ---
def _wnd_perp(r):
    rp = np.maximum(1.0 - r, 0.0)
    return rp**4 * (4*r + 1)
def _wnd_perp_d(r):
    """w, w'/r (pour éviter /r), w'' ; formes régularisées."""
    rp = np.maximum(1.0 - r, 0.0)
    w   = rp**4*(4*r+1)
    w1  = -20*rp**3*r                 # w'(r) = -20 r (1-r)^3
    w2  = -20*rp**2*(1 - 4*r)         # w''(r) = -20(1-r)^2(1-4r)
    return w, w1, w2


def plateau_ridge_atom(X, c, a_dir, sigma_a, sigma_perp, p_plateau=0.0,
                       weights=None):
    """φ et dérivées (grad, hess) d'UN atome plateau-ridge sur les points X.
    X:(n,d)  c:(d,)  a_dir:(d,) unitaire  sigma_a, sigma_perp: scalaires.
    Retourne phi:(n,), grad:(n,d), hess:(n,d,d) (ordre 3 non fourni : w3=0)."""
    n, d = X.shape
    a = a_dir / (np.linalg.norm(a_dir) + 1e-30)
    dx = X - c[None, :]                       # (n,d)
    ta = dx @ a                                # coord le long de a : (n,)
    perp = dx - np.outer(ta, a)                # composante perpendiculaire (n,d)
    rp2 = np.sum(perp**2, axis=1) / sigma_perp**2
    rp = np.sqrt(rp2)                          # rayon perp normalisé (n,)

    t = ta / sigma_a
    rho, rho1, rho2 = _plateau_prof_d(t, p_plateau)   # profil en a
    w, w1r, w2 = _wnd_perp_d(rp)                        # Wendland perp

    phi = rho * w

    # gradient : ∂φ = (∂ρ) w + ρ (∂w)
    #   ∂t/∂x = a/σ_a  ->  ∇(ρ) = rho1 * a/σ_a
    grad_rho = (rho1 / sigma_a)[:, None] * a[None, :]
    #   ∇(w) : w dépend de rp = ‖perp‖/σ_perp. ∇rp = perp/(σ_perp² rp) (dans perp)
    #   ∇w = w'(rp) ∇rp = w1r * perp / (σ_perp² rp)  ; régularisé via w1r/rp fini
    inv = np.where(rp > 1e-9, 1.0/(rp*sigma_perp**2), 0.0)
    grad_w = (w1r * inv)[:, None] * perp
    grad = grad_rho * w[:, None] + rho[:, None] * grad_w

    hess = None
    need_hess = weights is None or weights.get(2,0.)!=0
    if need_hess:
        # Hessienne = ρ'' term (a a^T) + cross + w'' term (perp perp^T) ...
        # H = w·Hρ + ρ·Hw + ∇ρ ∇w^T + ∇w ∇ρ^T
        aaT = np.outer(a, a)
        Hrho = (rho2 / sigma_a**2)[:, None, None] * aaT[None, :, :]
        # Hw : hessienne de W(rp). Standard : Hw = (w''- w'/rp)/rp² · perp perp^T /σ⁴
        #      + (w'/rp)/σ² · (I - a a^T)   (projection perp)
        Iperp = (np.eye(d) - aaT)[None, :, :]
        w1_over = np.where(rp > 1e-9, w1r/rp, 0.0)     # w'(rp)/rp fini en 0
        coef_pp = np.where(rp > 1e-9, (w2 - w1_over)/np.maximum(rp2,1e-18), 0.0)
        pp = np.einsum('ni,nj->nij', perp, perp)
        Hw = (coef_pp/sigma_perp**4)[:, None, None]*pp + (w1_over/sigma_perp**2)[:,None,None]*Iperp
        cross = np.einsum('ni,nj->nij', grad_rho, grad_w) \
              + np.einsum('ni,nj->nij', grad_w, grad_rho)
        hess = w[:,None,None]*Hrho + rho[:,None,None]*Hw + cross
    return phi, grad, hess


def atom_Hnorm2(X_cloud, c, a_dir, sigma_a, sigma_perp, p_plateau, weights):
    """‖atome‖²_H estimée sur X_cloud (ordres 1,2 ; w3=0)."""
    phi, grad, hess = plateau_ridge_atom(X_cloud, c, a_dir, sigma_a, sigma_perp,
                                         p_plateau, weights)
    tot = 0.0
    w0 = weights.get(0,0.)
    if w0: tot += w0*np.mean(phi**2)
    w1 = weights.get(1,0.)
    if w1: tot += w1*np.mean(np.sum(grad**2, axis=1))
    w2 = weights.get(2,0.)
    if w2 and hess is not None: tot += w2*np.mean(np.sum(hess**2, axis=(1,2)))
    return tot


# ══════════════════════════════════════════════════════════════════════════════
# SÉLECTION du meilleur a par centre (tirage sur S³) + assemblage d'un expert
# ══════════════════════════════════════════════════════════════════════════════
def best_a_for_center(X_cloud_local, c, sigma_a, sigma_perp, p_plateau,
                      weights, n_orient, rng):
    """Tire n_orient directions a sur la sphère S^{d-1}, garde celle de norme H
    MINIMALE de l'atome sur le cloud local (atome le plus 'plat' = aligné variété)."""
    d = X_cloud_local.shape[1]
    A = rng.normal(size=(n_orient, d))
    A /= np.linalg.norm(A, axis=1, keepdims=True)
    best_h = np.inf; best = A[0]
    for a in A:
        h = atom_Hnorm2(X_cloud_local, c, a, sigma_a, sigma_perp, p_plateau, weights)
        if h < best_h:
            best_h = h; best = a
    return best, best_h


def build_ridge_expert(centers, X_cloud, sigma_a, sigma_perp, p_plateau,
                       weights, n_orient, rng, n_local=300):
    """Un expert = un dictionnaire d'atomes plateau-ridge, un par centre, chacun
    avec son meilleur a (tirage) à sigma_a fixé (hyperparamètre de l'expert).
    Retourne : liste de (c, a, sigma_a, sigma_perp) pour évaluer features/dérivées."""
    atoms = []
    for c in centers:
        d2 = np.sum((X_cloud - c)**2, axis=1)
        loc = X_cloud[np.argsort(d2)[:min(n_local, len(X_cloud))]]
        a, _ = best_a_for_center(loc, c, sigma_a, sigma_perp, p_plateau,
                                 weights, n_orient, rng)
        atoms.append((c.copy(), a.copy(), sigma_a, sigma_perp))
    return atoms


def ridge_features(X, atoms, p_plateau):
    """Matrice (n, n_atoms) des valeurs des atomes."""
    n = len(X); k = len(atoms)
    A = np.zeros((n, k))
    for j, (c, a, sa, sp) in enumerate(atoms):
        A[:, j] = plateau_ridge_atom(X, c, a, sa, sp, p_plateau, {2:0})[0]
    return A


def ridge_derivs(X, atoms, p_plateau, weights):
    """Dérivées empilées : phi (n,k), grad (n,k,d), hess (n,k,d,d)."""
    n = len(X); k = len(atoms); d = X.shape[1]
    PHI = np.zeros((n, k)); GRAD = np.zeros((n, k, d)); HESS = np.zeros((n, k, d, d))
    for j, (c, a, sa, sp) in enumerate(atoms):
        phi, g, h = plateau_ridge_atom(X, c, a, sa, sp, p_plateau, weights)
        PHI[:, j] = phi; GRAD[:, j] = g
        if h is not None: HESS[:, j] = h
    return PHI, GRAD, HESS
# ─── RÉGLAGES (jouer ici) ─────────────────────────────────────────────────────
CFG = {
    "weights":     {0: 0.01, 1: 1.0, 2: 0.1, 3: 0.0},  # w3=0 pour des plans
    "n_centres":   800,     # centres par expert
    "sigma_a_list":[0.1, 0.3, 0.5],  # un expert par valeur (hyperparamètre |σ| normal)
    "sigma_perp":  1.0,     # extension tangentielle (large = constant le long des plans)
    "p_plateau":   0.1,     # largeur du plat central (tolérance points un peu à côté)
    "n_orient":    120,     # orientations tirées par centre pour trouver a
    "a_mode":      "tirage",# "tirage" (min norme H) | "impose" (a = a_true, validation)
    "qp_margin":   0.1,     # marge dure y_i u(x_i) >= marge
    "band_lo":     1e-4,    # bande spectrale : garde les modes s_max*band_lo < s < s_max*band_hi
    "band_hi":     1e3,
    "n_G":         600,     # points cloud pour estimer la Gram Sobolev
    "n_local":     250,     # voisinage local pour chercher a
    "seed":        1,
}

W = CFG["weights"]

# ─── QP en bande spectrale : min ‖u‖²_H s.c. diag(y)Ac >= marge ───────────────
def solve_qp_band(A, y, G, margin, band_lo, band_hi):
    s, V = np.linalg.eigh(G)
    smax = s.max()
    keep = (s > smax*band_lo) & (s < smax*band_hi)
    if keep.sum() == 0:
        return None, False, 0
    B = V[:, keep]; sB = s[keep]; Gd = np.diag(sB)
    A_r = A @ B
    con = LinearConstraint(np.diag(y) @ A_r, lb=margin, ub=np.inf)
    c0 = np.linalg.lstsq(A_r, 1.5*margin*y, rcond=None)[0]
    res = minimize(lambda al: al@Gd@al, c0, jac=lambda al: 2*Gd@al,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 800, 'ftol': 1e-11})
    coef = B @ res.x
    marge_eff = float(np.min(y * (A_r @ res.x)))
    feas = marge_eff >= margin - 1e-3
    return coef, feas, int(keep.sum())

def expert_G(atoms, X_cloud, rng):
    idx = rng.choice(len(X_cloud), min(CFG["n_G"], len(X_cloud)), replace=False)
    PHI, GRAD, HESS = ridge_derivs(X_cloud[idx], atoms, CFG["p_plateau"], W)
    nG = len(idx)
    G = W.get(0,0)*(PHI.T@PHI)/nG + W.get(1,0)*np.einsum('xik,xjk->ij',GRAD,GRAD)/nG
    if W.get(2,0): G += W[2]*np.einsum('xikl,xjkl->ij',HESS,HESS)/nG
    return G

# ─── PHASE 1 : un expert par σ_a ──────────────────────────────────────────────
print("\n" + "="*64)
print(f"WENDLAND CALIBRÉS  (a_mode={CFG['a_mode']}, {len(CFG['sigma_a_list'])} experts)")
print("="*64)
rng = np.random.default_rng(CFG["seed"])
experts = []   # (atoms, F_train_col, F_test_col)
F_tr_cols = []; F_te_cols = []; expert_info = []

for sa in CFG["sigma_a_list"]:
    cen = X_all[rng.choice(N_all, CFG["n_centres"], replace=False)]
    if CFG["a_mode"] == "impose":
        atoms = [(c.copy(), a_true.copy(), sa, CFG["sigma_perp"]) for c in cen]
        align = 1.0
    else:
        atoms = build_ridge_expert(cen, X_all, sa, CFG["sigma_perp"],
                                   CFG["p_plateau"], W, CFG["n_orient"], rng,
                                   n_local=CFG["n_local"])
        align = np.mean([abs(a @ a_true) for (_,a,_,_) in atoms])

    A_tr = ridge_features(X_train, atoms, CFG["p_plateau"])
    A_te = ridge_features(X_test,  atoms, CFG["p_plateau"])
    G = expert_G(atoms, X_all, rng)
    coef, feas, r = solve_qp_band(A_tr, yH_train, G, CFG["qp_margin"],
                                  CFG["band_lo"], CFG["band_hi"])
    if coef is None:
        print(f"  σ_a={sa}: bande spectrale vide"); continue
    f_tr = A_tr @ coef; f_te = A_te @ coef
    acc_tr = np.mean(np.sign(f_tr) == np.sign(yH_train))
    acc_te = np.mean(np.sign(f_te) == np.sign(yH_test))
    nH = np.sqrt(max(coef @ G @ coef, 0))
    print(f"  σ_a={sa:.2f} | |a·a_true|={align:.3f} | QP faisable={feas} r={r} | "
          f"‖u‖_H={nH:.3f} | acc_tr={acc_tr:.2f} acc_te={acc_te:.3f}")
    F_tr_cols.append(f_tr); F_te_cols.append(f_te)
    expert_info.append((atoms, G, coef))

# ─── PHASE 2 : combiner les experts (leurs sorties) par QP ────────────────────
if len(F_tr_cols) >= 2:
    Ftr = np.column_stack(F_tr_cols); Fte = np.column_stack(F_te_cols)
    # Gram H des sorties d'experts : approx par leurs normes (diagonale) — ici on
    # combine simplement au sens de la marge, G2 = identité (chaque expert normalisé).
    G2 = np.eye(Ftr.shape[1])
    a2, feas2, r2 = solve_qp_band(Ftr, yH_train, G2, CFG["qp_margin"], 1e-9, 1e9)
    if a2 is not None:
        fte = Fte @ a2
        acc = np.mean(np.sign(fte) == np.sign(yH_test))
        print(f"\n  PHASE 2 (combinaison QP) : faisable={feas2}  acc_te={acc:.3f}  alpha={a2.round(2)}")

print("\n(Comparaison : RBF ~0.95 sur ce problème ; hasard = 0.50)")


Génération du cloud sur {Q=0} = {P_sep=0} ∪ {P_sep=ε} (projection exacte, 50/50)...
  X_train:(10, 4)  X_test:(5000, 4)  X_unlabeled:(3990, 4)
  train : plan0=5  planε=5  max|résidu projection|=1.24e-14
  test  : plan0=2500  planε=2500  max|résidu projection|=2.49e-14
  y_train: 5 × (−2)  |  5 × (+2)   (labels ±a_label, décision sign(f))
cloud prêt. a_true = [-0.69   0.661 -0.18  -0.234]
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

WENDLAND CALIBRÉS  (a_mode=tirage, 3 experts)


/tmp/ipykernel_1561/243429649.py:326: RuntimeWarning: divide by zero encountered in divide
  inv = np.where(rp > 1e-9, 1.0/(rp*sigma_perp**2), 0.0)
/tmp/ipykernel_1561/243429649.py:340: RuntimeWarning: invalid value encountered in divide
  w1_over = np.where(rp > 1e-9, w1r/rp, 0.0)     # w'(rp)/rp fini en 0


  σ_a=0.10 | |a·a_true|=0.416 | QP faisable=False r=111 | ‖u‖_H=0.000 | acc_tr=0.00 acc_te=0.000


/tmp/ipykernel_1561/243429649.py:326: RuntimeWarning: divide by zero encountered in divide
  inv = np.where(rp > 1e-9, 1.0/(rp*sigma_perp**2), 0.0)
/tmp/ipykernel_1561/243429649.py:340: RuntimeWarning: invalid value encountered in divide
  w1_over = np.where(rp > 1e-9, w1r/rp, 0.0)     # w'(rp)/rp fini en 0


  σ_a=0.30 | |a·a_true|=0.422 | QP faisable=False r=105 | ‖u‖_H=0.000 | acc_tr=0.00 acc_te=0.000


/tmp/ipykernel_1561/243429649.py:326: RuntimeWarning: divide by zero encountered in divide
  inv = np.where(rp > 1e-9, 1.0/(rp*sigma_perp**2), 0.0)
/tmp/ipykernel_1561/243429649.py:340: RuntimeWarning: invalid value encountered in divide
  w1_over = np.where(rp > 1e-9, w1r/rp, 0.0)     # w'(rp)/rp fini en 0
